<a href="https://colab.research.google.com/github/irafilina1990-sudo/Tarefa_Database/blob/main/movie_example_1_(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

In [3]:
# загрузим исходные данные (датасет) в датафрейм (основной объект пайтон библиотеки пандас для данных)

df = pd.read_csv("MoviesOnStreamingPlatforms.csv")

#смотрим общую инф-ю о данных

print(df.info())

df.head(5)

FileNotFoundError: [Errno 2] No such file or directory: 'MoviesOnStreamingPlatforms.csv'

In [ ]:
print(df.columns.tolist())

# приведем данные к консистентной форме

In [ ]:
df.drop(columns=["Unnamed: 0", "ID", "Type"], inplace = True)

In [ ]:
df.head(5)


In [ ]:
df.info()

In [ ]:
# обработаем столбец age (заполним Null [пустые значения] на 'Unknown')

df['Age'] = df['Age'].fillna('Unknown')
print(df.info())

In [ ]:
df.head()

In [ ]:
# обработка столбца "рейтинг" - убираем деление на сто, так как не несет смысловой нагрузки для анализа

df['Rotten Tomatoes'] = df['Rotten Tomatoes'].str.replace('/100', '')
print(df.info())
df.head()

In [ ]:
#преобразуем к числовому типу данных колонку рейтинг

df['Rotten Tomatoes'] = pd.to_numeric(df['Rotten Tomatoes'], errors='coerce')
print(df.info())
df.head()

In [ ]:
# пустые ячейки в столбце рейтинга заполняем медианным значением рейтинга (устойчив к выбросам)

median_rating = df['Rotten Tomatoes'].median()

print(f"Медианное значение Rotten Tomatoes (Рейтинга)", median_rating)


In [ ]:
df['Rotten Tomatoes'] = df['Rotten Tomatoes'].fillna(median_rating)

print(df.info())
df.head()

In [ ]:
# Data Transform - это уже инструмент анализа.

Разведочный анализ:
рассмотрим:
1) к-во фильмов на каждой платформе
2) распределение по годам выпуска
3) распределение оценок качества (рейтинг)
4) Пересечения между платформами (сколько фильмов доступны на нескольких платфомах)

In [ ]:
# к-во фильмов на каждой платформе

platform_counts = {
    'Netflix': df['Netflix'].sum().item(),
    'Hulu' : df['Hulu'].sum().item(),
    'Prime Video' : df['Prime Video'].sum().item(),
    'Disney+': df['Disney+'].sum().item()

}

print(f"Количество фильмов на каждой платформе: \n\n {platform_counts}")

In [ ]:
platform_counts.keys()

In [ ]:
platform_counts.values()

In [ ]:
# визуализируем данные

import matplotlib.pyplot as plt

# зададим размер графика

plt.figure(figsize=(5,3))
plt.bar(platform_counts.keys(), platform_counts.values(), color = ['green', 'red', 'purple', 'blue'])

plt.title('Количество фильмов на каждой платформе')
plt.ylabel ('Количество фильмов')
plt.show()

Разница объясняется тем, что у Нетфликса и Прайм видео общирные библиотеки фильмов, а Дисней специализируется на своих фильмах и имеет ограниченный каталог


распределение по годам выпуска


In [ ]:
age_counts = df['Age'].value_counts()
print(f"Распределение по возрастам: \n\n {age_counts}")

In [ ]:
# категории

print(age_counts.index)

In [ ]:
# значения
print(age_counts.values)

In [ ]:
# видуализируем распределение по возрастному ограничению
plt.figure(figsize=(7,4))
plt.bar(age_counts.index, age_counts.values, color = ['green', 'red', 'purple', 'blue', 'black'])

plt.title('Количество фильмов по возрастам')
plt.ylabel ('Количество фильмов')
plt.show()

Главный вывод: самая большая категория — Unknown. Это означает, что в данных много фильмов без указанного возрастного рейтинга.

Содержательный вывод нужно формулировать осторожно. Нельзя сказать, что каталог в основном состоит из фильмов без рейтинга. Корректнее сказать: в датасете для значительной части фильмов возрастной рейтинг не указан.

Если исключить Unknown, наиболее заметна категория 18+, затем идут 7+ и 13+.

In [ ]:
df['Decade'] = (df['Year']//10)*10
decade_counts = df['Decade'].value_counts().sort_index()
print(f"Количество фильмов на каждой платформе: \n\n {decade_counts}")

In [ ]:
import seaborn as sns

In [ ]:
plt.figure(figsize=(12,5))
plt.bar(decade_counts.index, decade_counts.values, color = 'green')

plt.title('Количество фильмов по десятилетиям')
plt.ylabel ('Количество фильмов')
plt.show()




In [ ]:
# Распределение по годам выпуска
# Добавим столбец десятилетия для наглядности
df['Decade'] = (df['Year'] // 10) * 10
decade_counts = df['Decade'].value_counts().sort_index()
print("Количество фильмов по десятилетиям:\n", decade_counts)

plt.figure(figsize=(10,4))
plt.bar(decade_counts.index.astype(str), decade_counts.values, color='gray')
plt.title("Распределение фильмов по десятилетиям выпуска")
plt.xlabel("Десятилетие")
plt.ylabel("Количество фильмов")
plt.show()

In [ ]:
ratings = df['Rotten Tomatoes']

print("Минимальный рейтинг:", ratings.min())
print("Максимальный рейтинг:", ratings.max())
print("Средний рейтинг:", ratings.mean())
print("Медиана:", ratings.median())

In [ ]:
plt.figure(figsize=(6,4))
plt.hist(ratings, bins=20, edgecolor='black')
plt.title("Распределение оценок Rotten Tomatoes")
plt.xlabel("Рейтинг Rotten Tomatoes")
plt.ylabel("Число фильмов")
plt.show()

In [ ]:
df['PlatformCount'] = df[['Netflix', 'Hulu', 'Prime Video', 'Disney+']].sum(axis=1)
platform_count_dist = df['PlatformCount'].value_counts().sort_index()
print(f"Количество фильмов на каждой платформе: \n {platform_count_dist}")

In [ ]:
coverage = df['PlatformCount']
A_films = coverage[coverage >= 3].count()
B_films = coverage[coverage == 2].count()
C_films = coverage[coverage == 1].count()

print(f"Охваты: \n")

print(f"A:  {A_films}")
print(f"B:  {B_films}")
print(f"C:  {C_films}")

In [ ]:
X_films = coverage[coverage == 4].count()
Y_films = coverage[(coverage == 2) | (coverage == 3)].count()
Z_films = coverage[coverage == 1].count()
print(f"Охваты: \n")

print(f"X:  {X_films}")
print(f"Y:  {Y_films}")
print(f"Z:  {Z_films}")

In [ ]:
labels_abc = ['A (3-4 платформы)', 'B (2 платформы)', 'C (1 платформа)']
counts_abc = [A_films, B_films, C_films]

labels_xyz = ['X (4 платформы)', 'Y (2-3 платформы)', 'Z (1 платформа)']
counts_xyz = [X_films, Y_films, Z_films]

fig, axes = plt.subplots(1, 2, figsize=(10,4))
axes[0].bar(labels_abc, counts_abc)
axes[0].set_title("ABC-анализ")
axes[0].set_ylabel("Число фильмов")

axes[1].bar(labels_xyz, counts_xyz)
axes[1].set_title("XYZ-анализ")

plt.tight_layout()
plt.show()

In [ ]:
movies = df.copy()

In [ ]:
movies['R_segment'] = pd.qcut(movies['Year'], q=3, labels=['Old', 'Medium', 'New'])

In [ ]:
print(f"A:  {labels}")